# 📹 Unidad 1: Hardware y Sensores
## Tema: Global Shutter vs. Rolling Shutter

¡Bienvenido al laboratorio de Hardware! 🔬

En la presentación vimos que existen dos tipos principales de sensores:
1.  **CCD (Global Shutter):** Toma la foto de golpe. Congela el movimiento.
2.  **CMOS (Rolling Shutter):** Escanea la imagen línea por línea de arriba a abajo.

El Rolling Shutter causa efectos extraños cuando las cosas se mueven rápido (como la hélice de un avión que parece doblada). ¡Vamos a simularlo programando!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### 1. Creando un objeto en movimiento rápido

Imagina una barra vertical que se mueve muy rápido de izquierda a derecha. 
Vamos a definir una función que nos diga dónde está la barra en un tiempo $t$.

In [ ]:
# Dimensiones de nuestra 'cámara'
ALTO = 200
ANCHO = 200

def obtener_escena_en_tiempo_t(t, velocidad=50):
    """
    Devuelve una imagen de la escena en el instante 't'.
    La barra se mueve horizontalmente según la velocidad.
    """
    imagen = np.zeros((ALTO, ANCHO))
    
    # Calculamos la posición X de la barra
    pos_x = int(t * velocidad)
    
    # Dibujamos la barra vertical (si está dentro de la imagen)
    if 0 <= pos_x < ANCHO - 10:
        imagen[:, pos_x:pos_x+10] = 1 # Barra blanca de 10px de ancho
        
    return imagen

# Probemos ver la escena en t=1.5
plt.imshow(obtener_escena_en_tiempo_t(t=1.5), cmap='gray')
plt.title("Escena congelada en un instante")
plt.axis('off')
plt.show()

### 2. Simulación: Global vs Rolling

Ahora viene lo interesante:

* **Global Shutter:** Captura TODOS los píxeles en el mismo instante $t_{foto}$.
* **Rolling Shutter:** Captura la fila 0 en $t$, la fila 1 en $t + 0.01$, la fila 2 en $t + 0.02$, etc.

In [ ]:
# --- GLOBAL SHUTTER ---
# Toma la foto en t = 2.0 segundos exactos
foto_global = obtener_escena_en_tiempo_t(t=2.0)

# --- ROLLING SHUTTER ---
foto_rolling = np.zeros((ALTO, ANCHO))

# Simulamos que el sensor tarda un poquito en leer cada línea
retardo_por_linea = 0.01 
tiempo_inicio = 1.0

for fila in range(ALTO):
    # El tiempo avanza mientras bajamos por las filas
    tiempo_actual = tiempo_inicio + (fila * retardo_por_linea)
    
    # Obtenemos cómo se ve el mundo en ESE micro-instante exacto
    escena_instante = obtener_escena_en_tiempo_t(tiempo_actual)
    
    # Copiamos SOLO esa fila a nuestra foto final
    foto_rolling[fila, :] = escena_instante[fila, :]

# --- RESULTADOS ---
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Global Shutter\n(Barra Recta)")
plt.imshow(foto_global, cmap='gray')

plt.subplot(1, 2, 2)
plt.title("Rolling Shutter\n(Efecto 'Jello' - Inclinado)")
plt.imshow(foto_rolling, cmap='gray')

plt.show()

### Análisis del Fenómeno

¿Ves cómo la barra en la imagen de la derecha sale **diagonal**?

Esto pasa porque cuando el sensor estaba leyendo las filas de arriba, la barra estaba a la izquierda. Cuando el sensor llegó a leer las filas de abajo (milisegundos después), la barra ya se había movido a la derecha.

**Conclusión:** 
El hardware no es perfecto. Si vas a diseñar un robot que se mueva rápido o inspeccione objetos veloces en una banda transportadora, **necesitas gastar más dinero en una cámara con Global Shutter**, o tus algoritmos verán formas deformadas.